In [3]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

In [5]:

# Assuming your dataframes are loaded and indexed by 'sample_id'
# df_microbiome: rows=samples, cols=species (already CLR-transformed)
# df_metabolome: rows=samples, cols=metabolites
microbiome_data = pd.read_csv('data/microbiome.csv', index_col=0)
metabolome_data = pd.read_csv('data/metabolome.csv', index_col=0)
metadata = pd.read_csv('data/metadata.csv', index_col=0)

microbiome_data.index.name = 'SampleID'
metabolome_data.index.name = 'SampleID'
metadata.index.name = 'SampleID'
# 1. Find samples that have BOTH modalities for training the baseline
is_zero_row_microbiome = microbiome_data.sum(axis=1) == 0
is_zero_row_metabolome = metabolome_data.sum(axis=1) == 0

common_samples = microbiome_data[~is_zero_row_microbiome].index.intersection(metabolome_data[~is_zero_row_metabolome].index)
len(common_samples)
print(f"Number of samples with both modalities: {len(common_samples)}")
X_microbiom_full = microbiome_data.loc[common_samples]
X_metabolom_full = metabolome_data.loc[common_samples]

# Preprocessing metadata using get_dummies for categorical variables
# metadata_processed = pd.get_dummies(metadata, columns=['CENTER_C', 'PATGROUPFINAL_C'], drop_first=True).astype(float)

# Merge with metadata to get disease status

metadata_common = metadata.loc[common_samples]
metadata_common_no_Denemark = metadata_common[metadata_common['CENTER_C'] != 'Danemark']
X_microbiom_full = X_microbiom_full.loc[metadata_common_no_Denemark.index]
X_metabolom_full = X_metabolom_full.loc[metadata_common_no_Denemark.index]

metadata_common_no_Denemark = metadata_common_no_Denemark.loc[metadata_common_no_Denemark.index]
metadata_common_no_Denemark = pd.get_dummies(metadata_common_no_Denemark, columns=['CENTER_C', 'PATGROUPFINAL_C'], drop_first=True).astype(float)
print(f"Number of samples after removing Denmark: {len(metadata_common_no_Denemark)}")


Number of samples with both modalities: 1042
Number of samples after removing Denmark: 757


In [6]:
# Split into training and validation sets to test performance
X_micro_train, X_micro_val, X_metabolome_train, X_metabolome_val = train_test_split(
    X_microbiom_full, X_metabolom_full, test_size=0.2, random_state=42
)

#Merge training data with metadata
X_micro_train_with_metadata = X_micro_train.merge(metadata_common_no_Denemark, left_index=True, right_index=True)
X_metabolome_train_with_metadata = X_metabolome_train.merge(metadata_common_no_Denemark, left_index=True, right_index=True)
X_micro_val_with_metadata = X_micro_val.merge(metadata_common_no_Denemark, left_index=True, right_index=True)
X_metabolome_val_with_metadata = X_metabolome_val.merge(metadata_common_no_Denemark, left_index=True, right_index=True)

# --- Direction A: Metabolome -> Microbiome ---
print("Training: Metabolome to Predict Microbiome...")
rf_to_micro = MultiOutputRegressor(RandomForestRegressor(n_estimators=100, random_state=42, max_features=0.33, n_jobs=-1))
rf_to_micro.fit(X_metabolome_train_with_metadata, X_micro_train)
# Predict on validation set
pred_micro_val = rf_to_micro.predict(X_metabolome_val_with_metadata)

# --- Direction B: Microbiome -> Metabolome ---
print("Training: Microbiome to Predict Metabolome...")
rf_to_metabolome = MultiOutputRegressor(RandomForestRegressor(n_estimators=100, random_state=42, max_features=0.33, n_jobs=-1))
rf_to_metabolome.fit(X_micro_train_with_metadata, X_metabolome_train)
# Predict on validation set
pred_metabolome_val = rf_to_metabolome.predict(X_micro_val_with_metadata)

Training: Metabolome to Predict Microbiome...
Training: Microbiome to Predict Metabolome...


In [12]:
# ### WHY ARE WE DOING THIS?

# # Find samples that ONLY have metabolome (missing microbiome)
# missing_micro_samples = metabolome_data.index.difference(microbiome_data.index)
# # Filter out Denmark samples because they lack the metadata required for prediction
# missing_micro_samples = missing_micro_samples.intersection(metadata_common_no_Denemark.index)

# if len(missing_micro_samples) > 0:
#     X_metabolome_only = metabolome_data.loc[missing_micro_samples]
#     X_metabolome_only_with_metadata = X_metabolome_only.merge(metadata_common_no_Denemark, left_index=True, right_index=True)
#     imputed_micro = rf_to_micro.predict(X_metabolome_only_with_metadata)
#     df_micro_imputed = pd.DataFrame(imputed_micro, index=missing_micro_samples, columns=microbiome_data.columns)
#     # Combine original and imputed
#     df_micro_final = pd.concat([X_micro_train, df_micro_imputed])

# # Find samples that ONLY have microbiome (missing metabolome)
# missing_metabolome_samples = microbiome_data.index.difference(metabolome_data.index)
# # Filter out Denmark samples because they lack the metadata required for prediction
# missing_metabolome_samples = missing_metabolome_samples.intersection(metadata_common_no_Denemark.index)

# if len(missing_metabolome_samples) > 0:
#     X_micro_only = microbiome_data.loc[missing_metabolome_samples]
#     X_micro_only_with_metadata = X_micro_only.merge(metadata_common_no_Denemark, left_index=True, right_index=True)
#     imputed_metabolome = rf_to_metabolome.predict(X_micro_only_with_metadata)
#     df_meta_imputed = pd.DataFrame(imputed_metabolome, index=missing_metabolome_samples, columns=metabolome_data.columns)
#     # Combine original and imputed
#     df_metabolome_final = pd.concat([X_metabolome_train, df_meta_imputed])

In [13]:
from scipy.spatial.distance import pdist, squareform
from scipy.stats import spearmanr

def compute_project_distance_matrix(micro_df, meta_df):
    """Replicates the project's ground truth distance matrix generation"""
    # 1. Align rows
    common_idx = micro_df.index.intersection(meta_df.index)
    micro_aligned = micro_df.loc[common_idx]
    metabolome_aligned = meta_df.loc[common_idx]
    print(micro_df.shape, meta_df.shape, micro_aligned.shape, metabolome_aligned.shape)  # Debug shapes

    # 2. Standard Scale individual modalities
    scaler_micro = StandardScaler()
    scaler_meta = StandardScaler()
    
    micro_scaled = scaler_micro.fit_transform(micro_aligned)
    metabolome_scaled = scaler_meta.fit_transform(metabolome_aligned)
    
    # 3. Combine features
    combined_features = np.hstack((micro_scaled, metabolome_scaled))
    
    # 4. PCA to 2 components
    pca = PCA(n_components=2)
    pca_coords = pca.fit_transform(combined_features)
    
    # 5. Euclidean distance matrix
    dist_matrix = squareform(pdist(pca_coords, metric='euclidean'))
    return dist_matrix

def mantel_test(mat1, mat2):
    """Simplified Mantel test using Spearman correlation of the upper triangles"""
    # Extract upper triangles to avoid symmetry and diagonal bias
    tri_rows, tri_cols = np.triu_indices(mat1.shape[0], k=1)
    vec1 = mat1[tri_rows, tri_cols]
    vec2 = mat2[tri_rows, tri_cols]
    
    correlation, p_value = spearmanr(vec1, vec2)
    print(f"p-value: {p_value:.8e}")
    return correlation

# --- EVALUATION ---

# 1. Ground Truth Distance Matrix (using actual validation data)
# Convert numpy predictions back to DataFrames for the function
df_pred_micro_val = pd.DataFrame(pred_micro_val, index=X_micro_val.index, columns=X_micro_val.columns)
df_pred_metabolome_val = pd.DataFrame(pred_metabolome_val, index=X_metabolome_val.index, columns=X_metabolome_val.columns)

gt_distance_matrix = compute_project_distance_matrix(X_micro_val, X_metabolome_val)

# 2. Predicted Distance Matrix (using predicted microbiome instead of real)
pred_distance_matrix_micro = compute_project_distance_matrix(df_pred_micro_val, X_metabolome_val)

# 3. Run the Mantel Test comparison
baseline_score = mantel_test(gt_distance_matrix, pred_distance_matrix_micro)
print(f"Baseline Mantel Correlation (Random Forest Imputation): {baseline_score:.4f}")

(152, 170) (152, 102) (152, 170) (152, 102)
(152, 170) (152, 102) (152, 170) (152, 102)
p-value: 0.00000000e+00
Baseline Mantel Correlation (Random Forest Imputation): 0.5195
